In [1]:
import numpy as np
import pandas as pd
from glob import glob
import networkx as nx
import matplotlib.pyplot as plt
import ipycytoscape

import pandas as pd
from pyvis.network import Network
from IPython.display import IFrame


DATA_RAW_FILE = "DATA_RAW_CLEANED_20251103_20251116_ANTOFAGASTA.parquet"
DATA_RAW_FILE = [x for x in glob('../**', recursive=True) if DATA_RAW_FILE in x][0]
DATA_RAW_FILE

'..\\DataCleaning\\DATA_RAW_CLEANED_20251103_20251116_ANTOFAGASTA.parquet'

## READING DATA RAW

In [57]:
DATA_RAW = pd.read_parquet(DATA_RAW_FILE)
NODES = pd.DataFrame( list(set(DATA_RAW.index.get_level_values(0).tolist() + DATA_RAW.index.get_level_values(1).tolist())) ).rename(columns={0: 'name'})

EDGES_ID = pd.DataFrame([x for x in range(1000, 1000+len(DATA_RAW))], index=DATA_RAW.index, columns=['ID'])
EDGES_I_INV = {x['ID']:(x['A'],x['B']) for x in EDGES_ID.reset_index().to_dict('records')}


In [59]:

def draw_nodes_pyvis(data_raw: pd.DataFrame, filename: str = 'graph.html', directed: bool = False):
    # 1. Prepare Data (Same logic as before)
    data = data_raw.mean(axis=1).to_frame().round(0).rename(columns={0: 'bw'})
    
    # Node Data
    nodes_list = list(set(data.index.get_level_values(0).astype(str).tolist() + 
                          data.index.get_level_values(1).astype(str).tolist()))
    nodes_data = pd.DataFrame(nodes_list, columns=["id"])
    
    # Assign Colors/Sizes
    nodes_data['color'] = 'blue'
    nodes_data['size'] = 15  # Pyvis uses different scaling, smaller is better
    
    # Red Nodes
    hr_mask = nodes_data['id'].str.match(r'.*hr.*')
    nodes_data.loc[hr_mask, 'color'] = 'red'
    nodes_data.loc[hr_mask, 'size'] = 30

    # Yellow Nodes
    mr_mask = nodes_data['id'].str.match(r'.*mr_.*')
    nodes_data.loc[mr_mask, 'color'] = '#FFD700' # Hex for yellow often works better
    nodes_data.loc[mr_mask, 'size'] = 25
    
    # Green Nodes
    id_mask = nodes_data['id'].str.match(r'\d+[isp]?_\d+$')
    nodes_data.loc[id_mask, 'color'] = 'green'
    nodes_data.loc[id_mask, 'size'] = 10

    nodes_dict = nodes_data.to_dict('records')
    
    # Edge Data
    edge_df = data.reset_index()
    edge_df.iloc[:, 0] = edge_df.iloc[:, 0].astype(str)
    edge_df.iloc[:, 1] = edge_df.iloc[:, 1].astype(str)
    edge_data = list(edge_df.itertuples(index=False, name=None))

    # 2. Build Network
    # cdn_resources='remote' ensures it works without local JS files
    net = Network(height='600px', width='100%', notebook=True, cdn_resources='remote', directed=False) 
                  
    
    for node in nodes_dict:
        net.add_node(node['id'], color=node['color'], size=node['size'], label=node['id'])

    for src, tgt, bw in edge_data:
        net.add_edge(
            src, 
            tgt, 
            title=f"Bandwidth: {bw}",  # Shows on hover
            label=str(int(bw)),             # Shows permanently on the line
            width=2,
            font={'align': 'middle', 'strokeWidth': 0, 'background': 'white'} # Makes text easier to read
        )

    # 3. Physics Options (Similar to 'cose')
    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=100, spring_strength=0.08)
    
    # 4. Save and Show
    # This writes a file 'graph.html' to your folder. Open this file to see the graph!
    net.show(filename) 
    return f"Graph saved to '{filename}'. Open this file to view."

# Run it
#draw_nodes_pyvis(DATA_RAW)

draw_nodes_pyvis(DATA_RAW, filename='graph.html')
#IFrame(src='graph.html', width='100%', height='750px')

graph.html


"Graph saved to 'graph.html'. Open this file to view."


## PATCHTST

### Time Series Forecasting -  Raw Data

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans




def replace_outliers_with_nan_kmeans(series, n_clusters=3, contamination=0.05, random_state=42):

    if not isinstance(series, pd.Series):
        raise TypeError("Input must be a pandas Series.")

    valid_data = series.dropna()
    
    if len(valid_data) == 0:
        return series 
        
    X = valid_data.values.reshape(-1, 1)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X)

    all_distances = kmeans.transform(X)
    dist_to_center = np.min(all_distances, axis=1)

    threshold = np.quantile(dist_to_center, 1 - contamination)
    outlier_mask_valid = dist_to_center > threshold
    outlier_indices = valid_data.index[outlier_mask_valid]

    series_clean = series.copy()
    
    if not pd.api.types.is_float_dtype(series_clean):
        series_clean = series_clean.astype(float)
        
    series_clean.loc[outlier_indices] = np.nan

    return series_clean

def fix_missing_points(series, limit=4):
    return series.interpolate(method='linear', limit=limit, limit_direction='forward')

def fill_nan_with_time_mean(series):

    if not isinstance(series, pd.Series):
        raise TypeError("Input must be a pandas Series.")
        
    filled_series = series.copy()

    if not isinstance(filled_series.index, pd.DatetimeIndex):
        try:
            filled_series.index = pd.to_datetime(filled_series.index)
        except Exception as e:
            raise ValueError("Series index must be convertible to DatetimeIndex.") from e

    time_means = filled_series.groupby([filled_series.index.hour, filled_series.index.minute]).transform('mean')

    filled_series = filled_series.fillna(time_means)

    return filled_series


    _Xs = StandardScaler().fit_transform(data.values.astype(float).reshape(-1, 1))

    _model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(_Xs)
    _centers = _model.cluster_centers_[_model.labels_]
    _distances = np.linalg.norm(_Xs - _centers, axis=1)

    tmp = pd.DataFrame({"c": _model.labels_, "d": _distances}, index=TS_RAND['in'].index)
    
    _stats = tmp.groupby("c")["d"].agg(
        median="median",
        mad=lambda v: np.median(np.abs(v - np.median(v))) + 1e-9
        )
    
    _z = (tmp["d"].to_numpy() - _stats.loc[tmp["c"], "median"].to_numpy()) / _stats.loc[tmp["c"], "mad"].to_numpy()

    _thr = np.quantile(_z[np.isfinite(_z)], pct)
    _is_outlier = _z >= _thr

    return _is_outlier

DATA_TS = DATA_RAW.copy()
DATA_TS.index = EDGES_ID['ID']
DATA_TS = DATA_TS.T
DATA_TS = DATA_TS.asfreq('5min')

DATA_TS = DATA_TS.apply(lambda x: replace_outliers_with_nan_kmeans(x))
DATA_TS = DATA_TS.apply(lambda x: fix_missing_points(x))
DATA_TS = DATA_TS.apply(lambda x: fill_nan_with_time_mean(x))


print(f"Shape of the time series: {DATA_TS.shape}")
DATA_TS.sample(10)

c:\Users\valdo\anaconda3\envs\models3.11.3\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Shape of the time series: (4032, 97)


ID,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,1087,1088,1089,1090,1091,1092,1093,1094,1095,1096
2025-11-04 19:25:00,144.049688,38.244644,115.058993,6.854896,49.840178,148.079318,10.917038,0.0,718.617285,995.190445,...,83.946002,9.766391,83.373770,233.381380,130.278890,33.252049,125.082960,123.064566,111.606581,17.576131
2025-11-08 20:25:00,142.675826,37.163370,191.492373,3.606077,38.536899,170.619362,8.441102,0.0,751.720512,962.733720,...,108.270468,14.761941,63.922672,188.013635,121.460913,40.631575,91.975455,96.631479,140.677433,24.930272
2025-11-09 21:50:00,123.428291,32.778065,203.151939,2.878427,36.779581,175.920771,11.730524,0.0,790.980276,1142.062822,...,126.097819,276.749175,66.977317,199.591552,139.607439,31.572083,94.273911,95.851892,140.479327,21.346328
2025-11-12 23:55:00,106.041383,17.363684,155.742018,5.351869,25.638777,163.788962,10.506456,0.0,689.227580,1142.770442,...,115.695853,260.615299,72.119830,202.615408,106.106729,28.353617,111.913208,114.522216,118.192848,27.709452
2025-11-11 08:35:00,97.865631,29.998263,199.254387,6.668080,48.595251,96.080240,12.513651,0.0,457.498374,880.947431,...,72.013549,125.177461,41.433877,141.210712,114.391505,93.996048,150.335031,155.997574,86.973231,14.216619
2025-11-12 22:00:00,135.504203,25.409609,167.816878,2.924299,36.814735,171.688773,12.330736,0.0,747.589004,1390.300662,...,124.710942,262.751418,74.053136,204.795398,146.682050,28.985666,107.668853,103.010910,127.642503,29.842707
2025-11-11 01:00:00,106.487840,15.465263,110.492010,0.613154,20.146464,157.956027,0.017467,0.0,569.807027,656.331679,...,73.245967,194.243926,47.665474,165.768296,88.655708,22.041688,80.988488,81.566158,101.811627,17.332360
2025-11-03 18:35:00,117.004831,30.257313,189.776737,8.123963,47.094748,133.031035,10.878149,0.0,685.600974,1062.013599,...,103.847220,12.069027,89.671475,192.203727,109.795492,25.827911,123.867191,129.584633,119.924507,16.093388
2025-11-13 09:15:00,69.589430,18.755259,138.781261,5.217966,30.324118,104.202555,14.260873,0.0,536.183630,905.667539,...,90.844979,144.287030,57.628190,148.670488,115.991322,175.016557,149.941003,149.310787,88.869051,17.649871
2025-11-03 01:20:00,96.985440,12.817260,92.979393,0.476639,17.998168,113.414904,7.097206,0.0,544.088677,599.969060,...,74.432654,6.486521,53.773040,186.954688,75.637925,14.249903,95.907804,96.897088,99.877179,20.822092


### SOME STATISTICS

In [18]:
print(f"The mean number of point by day is {DATA_TS.resample('D').count().iloc[:,0].mean().round().astype(int)}")
print(f"The mean number of point by week is {DATA_TS.resample('W').count().iloc[:,0].mean().round().astype(int)}")

The mean number of point by day is 288
The mean number of point by week is 2016


# PATHTST

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

### PARAMETERS

- Date init: 2025-11-03
- Date end: 2025-11-16
- Frequency: 5 min
- Windows of training: 3 days
- Window of prediction: 1 day
- Channels: 96 Time Series

In [21]:
# TRAINING PARAMS
BATCH_SIZE = 32
EPOCHS = 20
LR = 0.001

# PREDICTION FOR A DAY
SEQ_LEN = 3*286           # 3 days
PRED_LEN = 1*286          # 1 day for prediction

CHANNELS =  DATA_TS.shape[1]
PATCH_LEN = 16
STRIDE = int(PATCH_LEN/2)
PATCH_NUM  = (SEQ_LEN - PRED_LEN) // STRIDE + 1 + 1

TIME_FEATURES = 4   #WEEK, DAY of WEEK, HOUR, MINUTE
D_MODEL = 128

### PREPARING DATA

### DATALOADERS

In [22]:
class TimeSeriesDataset(Dataset):
    """
    Converts a Pandas DataFrame into sliding windows for training.
    Extracts time features (Hour, Day, Month, DayOfWeek) from the index.
    """
    def __init__(self, dataframe: pd.DataFrame, seq_len: int, pred_len: int):
        self.seq_len = seq_len
        self.pred_len = pred_len
        
        # Ensure index is datetime
        if not isinstance(dataframe.index, pd.DatetimeIndex):
            dataframe.index = pd.to_datetime(dataframe.index)
            
        # 1. Extract Data Values (Normalized typically happening outside or via RevIN inside model)
        self.data = dataframe.values.astype(np.float32)
        
        # 2. Extract Time Features from Index
        # We normalize these to 0-based indices for Embedding layers
        df_stamp = dataframe.index
        self.time_features = np.stack([
            df_stamp.minute.values,
            df_stamp.hour.values,
            df_stamp.dayofweek.values,     # 0-6
            df_stamp.isocalendar().week.values.astype(np.int32) - 1, 
        ], axis=1).astype(np.int32)
        
    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1
    
    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + self.seq_len
        r_begin = s_end
        r_end = r_begin + self.pred_len
        
        # Input Sequence
        seq_x = self.data[s_begin:s_end]
        # Input Time Features
        seq_x_mark = self.time_features[s_begin:s_end]
        
        # Target Sequence
        seq_y = self.data[r_begin:r_end]
        
        return torch.tensor(seq_x), torch.tensor(seq_x_mark), torch.tensor(seq_y)


DATA_DS = TimeSeriesDataset(DATA_TS, seq_len=SEQ_LEN, pred_len=PRED_LEN)
DATA_DL = DataLoader(DATA_DS, batch_size=BATCH_SIZE, shuffle=True)


print(f"Total samples: {len(DATA_DS)}")

ts_data, time_data, y = DATA_DS[0]

print(f"Time Serie Shape: {ts_data.shape}")
print(f"Time Features Shape: {time_data.shape}")
print(f"Target Shape: {y.shape}")

Total samples: 2889
Time Serie Shape: torch.Size([858, 97])
Time Features Shape: torch.Size([858, 4])
Target Shape: torch.Size([286, 97])


### MODEL

In [23]:
class RevIN(nn.Module):
    """Reversible Instance Normalization to handle distribution shift."""
    def __init__(self, num_features: int, eps=1e-5, affine=True):
        super(RevIN, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if self.affine:
            self._init_params()

    def _init_params(self):
        self.affine_weight = nn.Parameter(torch.ones(self.num_features))
        self.affine_bias = nn.Parameter(torch.zeros(self.num_features))

    def forward(self, x, mode:str):
        if mode == 'norm':
            self._get_statistics(x)
            x = self._normalize(x)
        elif mode == 'denorm':
            x = self._denormalize(x)
        return x

    def _get_statistics(self, x):
        dim2reduce = tuple(range(1, x.ndim-1))
        self.mean = torch.mean(x, dim=dim2reduce, keepdim=True).detach()
        self.stdev = torch.sqrt(torch.var(x, dim=dim2reduce, keepdim=True, unbiased=False) + self.eps).detach()

        #self.register_buffer('mean', self.mean)
        #self.register_buffer('stdev', self.stdev)

    def _normalize(self, x):
        x = x - self.mean
        x = x / self.stdev
        if self.affine:
            x = x * self.affine_weight + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.affine:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
        x = x * self.stdev
        x = x + self.mean
        return x
        
class PatchingLayer(nn.Module):
    """
    Segments the time series into subseries-level patches.
    Input: (Batch, Seq_Len, Channels) or (Batch * Channels, Seq_Len, 1)
    Output: (Batch * Channels, Num_Patches, Patch_Len)
    """
    def __init__(self, patch_len: int, stride: int):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride

    def forward(self, x):
        # x shape: [Batch, Seq_Len, 1] (After Channel Independence Reshape)
        # padding logic to ensure we cover the whole sequence
        
        padding_needed = self.stride - (x.shape[1] - self.patch_len) % self.stride
        if padding_needed == self.stride: 
            padding_needed = 0
            
        # Pad the LAST value of the sequence
        if padding_needed > 0:
            last_val = x[:, -1:, :]
            x_pad = torch.cat([x, last_val.repeat(1, padding_needed, 1)], dim=1)
        else:
            x_pad = x

        # Unfold: Create patches
        # [Batch, Seq_Len, 1] -> [Batch, Num_Patches, Patch_Len]
        # We use unfold on dimension 1
        patches = x_pad.unfold(dimension=1, size=self.patch_len, step=self.stride)
        
        # Current shape: [Batch, Num_Patches, 1, Patch_Len]
        # We squeeze the channel dim (which is 1 due to Channel Independence)
        patches = patches.squeeze(2) 
        
        return patches
        
class TemporalEmbedding(nn.Module):
    """
    Embeds Minute, Hour, DayOfWeek, and Week based on the TimeSeriesDataset output.
    """
    def __init__(self, d_model, embed_type='fixed', freq='h'):
        super(TemporalEmbedding, self).__init__()

        # Approximate cardinalities

        # Cardinalities based on your Dataset:
        # Minute: 0-59 (size 60)
        # Hour: 0-23 (size 24)
        # DayOfWeek: 0-6 (size 7)
        # Week: 1-53 (size 54 to be safe)

        self.minute_embed = nn.Embedding(60, d_model)
        self.hour_embed = nn.Embedding(24, d_model)
        self.weekday_embed = nn.Embedding(7, d_model)
        self.week_embed = nn.Embedding(54, d_model)
    
    def forward(self, x):
        # x shape: [Batch, Seq_Len, 4] -> [Hour, Day, Month, Weekday]
        x = x.long()
        
        minute_x = self.minute_embed(x[:, :, 0])
        hour_x   = self.hour_embed(x[:, :, 1])
        weekday_x = self.weekday_embed(x[:, :, 2])
        week_x    = self.week_embed(x[:, :, 3])
        
        return minute_x + hour_x + weekday_x + week_x
        
class PatchTSTEmbedding(nn.Module):
    """
    Aggregates Value Embedding (Patch Projection), Position Embedding,
    and Time Embeddings.
    """
    def __init__(self, patch_len, d_model, num_patches, dropout=0.1):
        super().__init__()
        
        # 1. Value Embedding: Linear projection of the patch
        self.value_projection = nn.Linear(patch_len, d_model)
        
        # 2. Position Embedding: Learnable position for patch sequence
        self.position_embedding = nn.Parameter(torch.randn(1, num_patches, d_model))
        
        # 3. Time Feature Embedding (Optional/Auxiliary)
        self.time_embedding = TemporalEmbedding(d_model)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, patches, x_mark_enc):
        """
        patches: [Batch*Channels, Num_Patches, Patch_Len]
        x_mark_enc: [Batch*Channels, Num_Patches, 4] (Downsampled time features)
        """
        # A. Value Embedding
        # [Batch, Num_Patches, Patch_Len] -> [Batch, Num_Patches, d_model]
        enc_out = self.value_projection(patches)
        
        # B. Position Embedding
        # Add learnable position
        enc_out = enc_out + self.position_embedding[:, :enc_out.size(1), :]
        
        # C. Time Embedding (Add temporal context)
        # Project time features and add
        if x_mark_enc is not None:
             time_enc = self.time_embedding(x_mark_enc)
             enc_out = enc_out + time_enc
             
        return self.dropout(enc_out)
        
class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1):
        super().__init__()
        
        # Standard PyTorch Transformer Encoder Layer
        # Includes Multihead Attention and FeedForward (Linear -> GELU -> Linear)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            dim_feedforward=d_ff, 
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True # Pre-norm is generally better for Time Series
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        return self.encoder(x)

class FlattenHead(nn.Module):
    def __init__(self, num_patches, d_model, target_len, head_dropout=0.0):
        super().__init__()
        self.head = nn.Sequential(
            nn.Flatten(start_dim=-2),
            nn.Linear(num_patches * d_model, target_len),
            nn.Dropout(head_dropout)
        )

    def forward(self, x):
        # x: [Batch, Num_Patches, d_model]
        # output: [Batch, Target_Len]
        return self.head(x)

class PatchTST(nn.Module):

    def __init__(self, 
                 num_input_channels, 
                 seq_len, 
                 pred_len, 
                 patch_len=16, 
                 stride=8, 
                 d_model=128, 
                 n_heads=4, 
                 n_layers=3, 
                 d_ff=256, 
                 dropout=0.1):
        super().__init__()
        
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_channels = num_input_channels
        self.patch_len = patch_len
        self.stride = stride
        
        # Calculate number of patches
        # Fix: Exact calculation matching PatchingLayer padding logic
        self.num_patches = (seq_len - patch_len) // stride + 1
        if (seq_len - patch_len) % stride != 0:
            self.num_patches += 1
        
        # 1. Instance Norm (RevIN)
        self.revin = RevIN(num_features=1, affine=True) # Applied per channel
        
        # 2. Patching
        self.patching_layer = PatchingLayer(patch_len, stride)
        
        # 3. Embeddings (Value + Position + Time)
        self.embedding_layer = PatchTSTEmbedding(patch_len, d_model, self.num_patches, dropout)
        
        # 4. Transformer Encoder
        self.encoder = TransformerBackbone(d_model, n_heads, d_ff, n_layers, dropout)
        
        # 5. Prediction Head
        self.head = FlattenHead(self.num_patches, d_model, pred_len)

    def forward(self, x, x_mark):
        """
        x: [Batch, Seq_Len, Num_Channels]
        x_mark: [Batch, Seq_Len, 4] (Time features)
        """
        B, L, M = x.shape
        
        # --- Channel Independence Logic ---
        # Reshape: [Batch, Seq_Len, Channels] -> [Batch * Channels, Seq_Len, 1]
        x = x.permute(0, 2, 1).reshape(B * M, L, 1)
        
        # Handle time features for Channel Independence (Repeat for each channel)
        # x_mark: [B, L, 4] -> [B*M, L, 4]
        x_mark = x_mark.repeat_interleave(M, dim=0)

        # 1. Instance Normalization
        x = self.revin(x, 'norm') # [B*M, L, 1]

        # 2. Patching
        # [B*M, L, 1] -> [B*M, Num_Patches, Patch_Len]
        x_patches = self.patching_layer(x)
        
        # Prepare Time Features for Patches (Downsample to patch granularity)
        # We take the time feature of the last time step in every patch
        pad_len = self.stride - (self.seq_len - self.patch_len) % self.stride
        if pad_len == self.stride: pad_len = 0
        
        # Pad marks similarly to data
        if pad_len > 0:
            last_mark = x_mark[:, -1:, :]
            x_mark_pad = torch.cat([x_mark, last_mark.repeat(1, pad_len, 1)], dim=1)
        else:
            x_mark_pad = x_mark
            
        # Unfold marks: [B*M, Num_Patches, Patch_Len, 4]
        # We select the last step of the patch as the "time" of the patch
        # Fix: Ensure indices are on the same device as input x
        patch_indices = torch.arange(0, self.num_patches, device=x.device) * self.stride + (self.patch_len - 1)
        patch_indices = torch.clamp(patch_indices, max=x_mark_pad.shape[1]-1).long()
        x_mark_patches = x_mark_pad[:, patch_indices, :] # [B*M, Num_Patches, 4]

        # 3. Embeddings (Value Proj + Pos + Time)
        # [B*M, Num_Patches, d_model]
        x_enc = self.embedding_layer(x_patches, x_mark_patches)
        
        # 4. Transformer Encoder
        enc_out = self.encoder(x_enc)
        
        # 5. Head
        out = self.head(enc_out) # [B*M, Pred_Len]
        
        # 6. Reverse Instance Normalization
        # Reshape back to apply Denorm: [B*M, Pred_Len, 1]
        out = out.unsqueeze(-1)
        out = self.revin(out, 'denorm')
        
        # Reshape back to [Batch, Pred_Len, Num_Channels]
        out = out.reshape(B, M, self.pred_len).permute(0, 2, 1)
        
        return out
   

## TRAINING PROCESS

In [64]:
MODEL = PatchTST(
    num_input_channels=CHANNELS,
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
    patch_len=PATCH_LEN,
    stride=STRIDE,
    d_model=D_MODEL,
    n_heads=4,
    n_layers=2
)


c:\Users\valdo\anaconda3\envs\models3.11.3\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [65]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
MODEL.to(device)

Using device: cuda


PatchTST(
  (revin): RevIN()
  (patching_layer): PatchingLayer()
  (embedding_layer): PatchTSTEmbedding(
    (value_projection): Linear(in_features=16, out_features=128, bias=True)
    (time_embedding): TemporalEmbedding(
      (minute_embed): Embedding(60, 128)
      (hour_embed): Embedding(24, 128)
      (weekday_embed): Embedding(7, 128)
      (week_embed): Embedding(54, 128)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): TransformerBackbone(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine

In [21]:
optimizer = optim.Adam(MODEL.parameters(), lr=LR)
criterion = nn.MSELoss()

In [ ]:
print("Starting Training...")
MODEL.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_x, batch_x_mark, batch_y in DATA_DL:
        batch_x = batch_x.to(device)         # [B, L, C]
        batch_x_mark = batch_x_mark.to(device) # [B, L, 4]
        batch_y = batch_y.to(device)         # [B, P, C]
        
        optimizer.zero_grad()
        
        # Forward
        outputs = MODEL(batch_x, batch_x_mark)
        
        # Loss Calculation
        loss = criterion(outputs, batch_y)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(DATA_DL)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")   

Starting Training...
Epoch [1/20], Loss: 96798.6979
Epoch [2/20], Loss: 6096.5152
Epoch [3/20], Loss: 4979.3396
Epoch [4/20], Loss: 4705.2770
Epoch [5/20], Loss: 4741.8582
Epoch [6/20], Loss: 5293.6877
Epoch [7/20], Loss: 5611.2687
Epoch [8/20], Loss: 4288.6426
Epoch [9/20], Loss: 3338.6009
Epoch [10/20], Loss: 2850.2385
Epoch [11/20], Loss: 2613.1446
Epoch [12/20], Loss: 2458.9854
Epoch [13/20], Loss: 2340.7856
Epoch [14/20], Loss: 2184.5326
Epoch [15/20], Loss: 2119.6744
Epoch [16/20], Loss: 1976.0694
Epoch [17/20], Loss: 1981.9283
Epoch [18/20], Loss: 1618.3921
Epoch [19/20], Loss: 1456.6928
Epoch [20/20], Loss: 1424.6497


In [83]:
# Define your file path
SAVE_PATH = "timeseries_model.pth"

# 1. Save the model weights
torch.save(MODEL.state_dict(), SAVE_PATH)

print(f"Model saved to {SAVE_PATH}")

Model saved to timeseries_model.pth


In [85]:
checkpoint = {
    'epoch': epoch,
    'model_state_dict': MODEL.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': total_loss,
    # If using RevIN, its internal mean/stdev are saved automatically 
    # as long as they are registered as buffers or parameters.
}

torch.save(checkpoint, "checkpoint.pth")

# EVALUATION OF PATCHTST

In [26]:
MODEL = PatchTST(
    num_input_channels=CHANNELS,
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
    patch_len=PATCH_LEN,
    stride=STRIDE,
    d_model=D_MODEL,
    n_heads=4,
    n_layers=2
)

state_dict = torch.load('timeseries_model_2Weeks.pth', map_location='cuda')
MODEL.load_state_dict(state_dict)
MODEL.to('cuda')
MODEL.eval()

PatchTST(
  (revin): RevIN()
  (patching_layer): PatchingLayer()
  (embedding_layer): PatchTSTEmbedding(
    (value_projection): Linear(in_features=16, out_features=128, bias=True)
    (time_embedding): TemporalEmbedding(
      (minute_embed): Embedding(60, 128)
      (hour_embed): Embedding(24, 128)
      (weekday_embed): Embedding(7, 128)
      (week_embed): Embedding(54, 128)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): TransformerBackbone(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine

In [ ]:

def plot_results(model, dataframe, seq_len, pred_len, sample_idx=0):
    """
    Plots the Input History, Ground Truth Future, and Model Prediction for a specific sample.
    """
    dataset = TimeSeriesDataset(dataframe, seq_len, pred_len)
    
    # Get a single sample
    input_x, input_x_mark, true_y = dataset[sample_idx]
    
    # Add batch dimension for the model
    input_x_batch = input_x.unsqueeze(0)        # [1, seq_len, channels]
    input_x_mark_batch = input_x_mark.unsqueeze(0) # [1, seq_len, 4]
    
    device = next(model.parameters()).device
    model.eval()
    
    with torch.no_grad():
        input_x_batch = input_x_batch.to(device)
        input_x_mark_batch = input_x_mark_batch.to(device)
        pred_y = model(input_x_batch, input_x_mark_batch)
    
    # Convert to numpy for plotting
    input_x = input_x.numpy()
    true_y = true_y.numpy()
    pred_y = pred_y.cpu().numpy().squeeze(0) # Remove batch dim
    
    # Plotting
    # --- Setup for Grid Layout ---
    num_cols = 3
    num_channels = input_x.shape[1]
    # Calculate rows needed (ceiling division)
    num_rows = (num_channels + num_cols - 1) // num_cols 

    # Create the subplots
    # Note: Increased width (15) to accommodate 3 columns
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows), sharex=False)

    # Flatten the axes array (makes it 1D so we can iterate over it easily)
    # If num_channels is small, axes might not be an array, so we ensure it is iterable
    if num_channels == 1:
        axes = np.array([axes])
    axes_flat = axes.flatten()

    # Time axis for plotting
    x_history = np.arange(seq_len)
    x_future = np.arange(seq_len, seq_len + pred_len)

    for i in range(num_channels):
        ax = axes_flat[i]
        
        # Plot History
        ax.plot(x_history, input_x[:, i], label='History', color='blue')
        
        # Plot Ground Truth
        ax.plot(x_future, true_y[:, i], label='Ground Truth', color='green', linestyle='dashed')
        
        # Plot Prediction
        ax.plot(x_future, pred_y[:, i], label='Prediction', color='red')
        
        ax.set_title(f"Channel {i}")
        ax.legend()
        ax.grid(True)
    # --- Clean up empty subplots ---
    # If num_channels isn't a perfect multiple of 3, hide the empty axes at the end
    for i in range(num_channels, len(axes_flat)):
        fig.delaxes(axes_flat[i])
    
    plt.xlabel("Time Steps")
    plt.tight_layout()
    plt.show()

plot_results(MODEL, DATA_TS, SEQ_LEN, PRED_LEN, sample_idx=1500)

In [32]:


from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_on(model, dataloader, criterion, device, plot_sample=True):
    """
    Evaluates the PatchTST model on the specified device (CPU or CUDA).
    
    Args:
        model: The trained PatchTST model.
        dataloader: Test/Validation DataLoader.
        criterion: Loss function (e.g., MSELoss).
        device: torch.device object (e.g., 'cuda' or 'cpu').
        plot_sample: Whether to plot a random forecast result.
        
    Returns:
        metrics: Dictionary with MSE, MAE, and Loss.
        preds: Numpy array of predictions (N, M, T).
        actuals: Numpy array of ground truth (N, M, T).
    """
    # 1. Move model to the selected device
    model.to(device)
    model.eval() 
    
    all_preds = []
    all_actuals = []
    total_loss = 0.0
    
    print(f"Starting evaluation on {device}...")
    
    # 2. Evaluation Loop (No Gradients)
    with torch.no_grad():
        for i, (batch_x, batch_x_time, batch_y) in enumerate(dataloader):
            # Move batch data to the specified device
            batch_x = batch_x.to(device)
            batch_x_time = batch_x_time.to(device)
            batch_y = batch_y.to(device)
            
            # Forward Pass
            preds = model(batch_x, batch_x_time)
            
            # Calculate Loss
            loss = criterion(preds, batch_y)
            total_loss += loss.item()
            
            # Store results
            # CRITICAL: We must detach and move to CPU before converting to numpy
            all_preds.append(preds.detach().cpu().numpy())
            all_actuals.append(batch_y.detach().cpu().numpy())

    # 3. Aggregate Results
    # Concatenate all batches into single arrays
    preds_np = np.concatenate(all_preds, axis=0)
    actuals_np = np.concatenate(all_actuals, axis=0)
    
    # 4. Calculate Metrics (Real-World Scale)
    mse = mean_squared_error(actuals_np.flatten(), preds_np.flatten())
    mae = mean_absolute_error(actuals_np.flatten(), preds_np.flatten())
    avg_loss = total_loss / len(dataloader)
    
    metrics = {
        'MSE': mse,
        'MAE': mae, 
        'Avg_Batch_Loss': avg_loss
    }
    
    print(f"\n--- Evaluation Results ---")
    print(f"Global MSE: {mse:.4f}")
    print(f"Global MAE: {mae:.4f}")
    print(f"Avg Loss:   {avg_loss:.4f}")
    
    # 5. Optional Visualization
    if plot_sample:
        _plot_forecast(preds_np, actuals_np)
        
    return metrics, preds_np, actuals_np

def _plot_forecast(preds, actuals):
    """
    Helper function to plot a random sample from the numpy arrays.
    """
    # Pick random sample and random router
    sample_idx = np.random.randint(0, preds.shape[0])
    router_idx = np.random.randint(0, preds.shape[1])
    
    pred_series = preds[sample_idx, router_idx, :]
    actual_series = actuals[sample_idx, router_idx, :]
    
    plt.figure(figsize=(10, 5))
    plt.plot(actual_series, label='Actual', marker='.', color='black')
    plt.plot(pred_series, label='Forecast', linestyle='--', color='red')
    plt.title(f"Sample {sample_idx} | Router {router_idx}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    
criterion = torch.nn.MSELoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

metrics, preds, actuals = evaluate_on(
    model=MODEL, 
    dataloader=DATA_DL, 
    criterion=criterion,
    device=device,  # Pass the device here
    plot_sample=False
)

Starting evaluation on cuda...

--- Evaluation Results ---
Global MSE: 1772.1531
Global MAE: 22.6507
Avg Loss:   1768.7686


# PREDICTION

In [62]:
def predict_future_dataframe(model, dataframe, seq_len, pred_len, device):
    """
    Predicts the next 'pred_len' steps using the last 'seq_len' rows of the dataframe.
    Adapts strictly to the TimeSeriesDataset logic.
    """
    model.eval()
    
    # 1. Validation
    if len(dataframe) < seq_len:
        raise ValueError(f"Dataframe is too short. Needs at least {seq_len} rows.")
    
    if not isinstance(dataframe.index, pd.DatetimeIndex):
        dataframe.index = pd.to_datetime(dataframe.index)

    # 2. Extract Last Window (History)
    # We grab the very last 'seq_len' rows
    last_window_df = dataframe.iloc[-seq_len:].fillna(method='ffill')

    # --- A. Process Raw Data (x) ---
    # Dataset logic: self.data = dataframe.values.astype(np.float32)
    # Shape: (L, M)
    last_values = last_window_df.values.astype(np.float32)

    
    # Model Expectation: (Batch, M, L)
    # We must TRANSPOSE: (L, M) -> (M, L) -> Add Batch (1, M, L)
    input_x = torch.tensor(last_values).unsqueeze(0).to(device)

    # --- B. Process Time Features (x_mark) ---
    # Dataset logic: Extract Minute, Hour, DayOfWeek, Week
    df_stamp = last_window_df.index
    
    time_features = np.stack([
        df_stamp.minute.values,
        df_stamp.hour.values,
        df_stamp.dayofweek.values,
        df_stamp.isocalendar().week.values.astype(np.int32) - 1, 
    ], axis=1).astype(np.int64) # Model usually expects Float32 for linear layers
    
    
    # Shape: (L, 4)
    # Model Expectation: (Batch, L, C_time) -> Just add Batch dim
    input_x_mark = torch.tensor(time_features).unsqueeze(0).to(device)

    # 3. Predict
    with torch.no_grad():
        # Input shapes are now:
        # x:      (1, M, seq_len)
        # x_mark: (1, seq_len, 4)
        preds = model(input_x, input_x_mark)
    
    # 4. Process Output
    # Output shape: (1, M, pred_len) -> Squeeze -> (M, pred_len) -> Transpose to (pred_len, M)
    forecast_values = preds.squeeze(0).cpu().numpy().T

    # 5. Generate Future Index
    last_date = dataframe.index[-1]
    # Infer frequency from the dataframe (e.g., 'H', '15T', 'D')
    freq = pd.infer_freq(dataframe.index)
    if freq is None: freq = '5T' # Fallback default
    offset = pd.to_timedelta(freq)

    future_dates = pd.date_range(start=last_date + offset, 
                                 periods=pred_len, 
                                 freq=freq)

  
    
    # 6. Return DataFrame
    return pd.DataFrame(forecast_values.T, index=future_dates, columns=dataframe.columns)


state_dict = torch.load('timeseries_model_2Weeks.pth', map_location='cuda')
MODEL = PatchTST(
            num_input_channels=CHANNELS,
            seq_len=SEQ_LEN,
            pred_len=PRED_LEN,
            patch_len=PATCH_LEN,
            stride=STRIDE,
            d_model=D_MODEL,
            n_heads=4,
            n_layers=2
        )
        



# 1. Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL.load_state_dict(state_dict)
MODEL.to(device)
MODEL.eval()

DATA_TS_PRED = predict_future_dataframe(
    model=MODEL, 
    dataframe=DATA_TS, 
    seq_len=SEQ_LEN, 
    pred_len=PRED_LEN, 
    device=device
)



DATA_TS_PRED = DATA_TS_PRED.T
DATA_TS_PRED.index = pd.MultiIndex.from_tuples([EDGES_I_INV[x] for x in DATA_TS_PRED.index])


draw_nodes_pyvis(DATA_TS_PRED, filename='graph_pred.html')

graph_pred.html


c:\Users\valdo\anaconda3\envs\models3.11.3\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
C:\Users\valdo\AppData\Local\Temp\ipykernel_17000\2978132895.py:17: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  last_window_df = dataframe.iloc[-seq_len:].fillna(method='ffill')


"Graph saved to 'graph_pred.html'. Open this file to view."